# NetGuard 前端重构设计草案（Vue 版）

## 1. 重构目标
- 技术栈从 `Streamlit` 迁移为 `Vue 3 + Element Plus + ECharts + FastAPI`。
- 第一阶段目标：先复现当前 Streamlit 的信息结构与页面逻辑。
- 第二阶段目标：在保持数据一致性的前提下提升可视化与交互体验。

## 2. 总体架构
- 前端：`Vue 3 + Vite + Element Plus + ECharts + Pinia + Vue Router`
- 后端：`FastAPI`（提供 REST + WebSocket）
- 数据层：继续使用 `SQLite(netguard_logs.db)`（后续可迁移 PostgreSQL）
- 数据来源：`realtime_predictions.jsonl -> backend_engine 写入 SQLite -> FastAPI 提供接口`

## 3. 刷新策略（改为 WebSocket 推送）
- 不再使用固定轮询作为主刷新手段。
- 后端新增 WebSocket 通道：`/ws/overview`、`/ws/alerts`、`/ws/ai`。
- 推送触发条件：
  - 有新告警写入 `traffic_alerts` 时推送增量。
  - 有新 AI 研判写入 `ai_insights` 时推送摘要。
  - 每隔固定心跳（如 10s）推送状态包，保证前端在线状态显示。
- 前端策略：
  - WebSocket 为主，REST 为兜底（首屏初始化/断线重连后补数据）。
  - 断线自动重连（指数退避），重连后调用 REST 补齐遗漏窗口。

## 4. 页面信息架构（复现现有逻辑）
- 全局安全态势
- 实时防御矩阵
- XAI 决策解释
- 模型效能评估

## 5. 页面草图细化

### 5.1 全局安全态势
- 顶部 4 个指标卡：总记录数、中高危数、unknown 数、安全评分
- 24h 攻防态势面积图（ECharts）
- 风险等级占比环图（ECharts）
- 威胁流向图（Sankey）
- AI 场景研判摘要卡（告警研判）
- AI 用户行为分析卡（已知类别行为画像）

### 5.2 实时防御矩阵
- 告警等级过滤器（low/medium/high）
- 实时告警表（Element Plus `el-table`）
- 高频源 IP 柱状图
- 表格按风险分/时间排序，支持关键词筛选

### 5.3 XAI 决策解释
- 告警样本选择器
- 指标卡：confidence、risk_score、distance、threshold
- 包级贡献图（Occlusion）
- 字节热力图（Grad-CAM）
- 证据 JSON 折叠查看

### 5.4 模型效能评估
- 指标卡：离线准确率、未知样本占比
- 雷达图：Baseline vs NetGuard
- 高维特征分离图（先 2D/3D 散点）
- AI 研判历史（含 `insight_type`: alert/behavior）

## 6. API 与 WS 设计草案

### 6.1 REST（初始化/补偿）
- `GET /api/overview`
- `GET /api/alerts?level=...&limit=...`
- `GET /api/xai/samples`
- `GET /api/xai/detail?id=...`
- `GET /api/ai/insights?type=alert|behavior`
- `GET /api/model/metrics`

### 6.2 WebSocket（实时推送）
- `WS /ws/overview`：总量、风险分布、趋势增量
- `WS /ws/alerts`：新告警事件流
- `WS /ws/ai`：新 AI 研判事件流

### 6.3 推送消息协议（示例）
```json
{
  "event": "alert_inserted",
  "ts": "2026-03-08 21:35:10",
  "payload": {
    "id": 1024,
    "alert_level": "high",
    "threat_category": "unknown_proxy_ood",
    "risk_score": 1.73
  }
}
```

## 7. 组件拆分建议
- `DashboardOverview.vue`
- `DefenseMatrix.vue`
- `XaiInspector.vue`
- `ModelEvaluation.vue`
- `AiInsightCard.vue`
- `BehaviorInsightCard.vue`
- `MetricCard.vue`
- `ChartCard.vue`

## 8. 状态管理与可用性
- Pinia Store：`overviewStore`、`alertStore`、`xaiStore`、`aiStore`
- WebSocket 状态机：`connecting / open / closed / retrying`
- 断线重连 + REST 补偿，防止丢包导致的界面断层

## 9. 实施顺序
1. 搭建 Vue 路由与页面骨架（4 个页面）
2. 接入 FastAPI REST，先完成静态复现
3. 接入 WebSocket 推送，替换轮询
4. 优化动效、主题、组件细节，完成挑战杯展示版本